# Multistage & Mission Architecture

Demonstrates the multistage/mission architecture as it is built up commit by commit (see `mission_multistage_design.md`). Each section below corresponds to one landed commit and is added to as the next commit lands.

**Landed so far:**
1. `Stage` — wraps a single-stage `Rocket`
2. `Deployable` — a carried payload released mid-flight
3. `MultiStageRocket` — composes a stage + its deployables into one flight-ready `Rocket`
4. `MultiStageRocket.flight_rocket` for more than one active stage (a booster with an inert sustainer riding on top)
5. `Mission` — orchestrates one `Flight` per vehicle configuration (single-stage degenerate case, and now a full two-stage mission: deterministic burnout+delay separation, state handoff, and sustainer ignition timing)


In [ ]:
from rocketpy import NoseCone, Rocket, SolidMotor
from rocketpy.rocket.multistage import Deployable, MultiStageRocket, Stage

## Building a base rocket

`Stage` and `MultiStageRocket` wrap ordinary `Rocket` objects — nothing new is needed to build one. This is the same Calisto rocket used throughout the RocketPy docs.

In [ ]:
Pro75M1670 = SolidMotor(
    thrust_source="../../data/motors/cesaroni/Cesaroni_M1670.eng",
    dry_mass=1.815,
    dry_inertia=(0.125, 0.125, 0.002),
    nozzle_radius=33 / 1000,
    grain_number=5,
    grain_density=1815,
    grain_outer_radius=33 / 1000,
    grain_initial_inner_radius=15 / 1000,
    grain_initial_height=120 / 1000,
    grain_separation=5 / 1000,
    grains_center_of_mass_position=0.397,
    center_of_dry_mass_position=0.317,
    nozzle_position=0,
    burn_time=3.9,
    throat_radius=11 / 1000,
    coordinate_system_orientation="nozzle_to_combustion_chamber",
)

calisto = Rocket(
    radius=127 / 2000,
    mass=14.426,
    inertia=(6.321, 6.321, 0.034),
    power_off_drag="../../data/rockets/calisto/powerOffDragCurve.csv",
    power_on_drag="../../data/rockets/calisto/powerOnDragCurve.csv",
    center_of_mass_without_motor=0,
    coordinate_system_orientation="tail_to_nose",
)
calisto.add_motor(Pro75M1670, position=-1.255)

print(f"calisto.dry_mass = {calisto.dry_mass:.4f} kg")
print(f"calisto.motor.burn_out_time = {calisto.motor.burn_out_time} s")

## `Stage`: wrapping a rocket as one stage of a vehicle

For now `Stage` is a thin wrapper: `dry_mass` and `burn_out_time` just forward to the wrapped `Rocket`'s own already-computed attributes. It becomes meaningful once `MultiStageRocket` composes several stages together (a later commit).

In [ ]:
booster = Stage(name="booster", rocket=calisto)

print(f"booster.dry_mass = {booster.dry_mass:.4f} kg")
print(f"booster.burn_out_time = {booster.burn_out_time} s")

assert booster.dry_mass == calisto.dry_mass
assert booster.burn_out_time == calisto.motor.burn_out_time

## `Deployable`: a payload carried and ejected mid-flight

A `Deployable` contributes only mass and inertia while attached. Its free-flight aerodynamics come from either a fully built `free_rocket`, or surfaces added one at a time with `add_surface()` — the two are mutually exclusive, and `add_surface()` requires `radius` to be set.

In [ ]:
payload = Deployable(
    name="payload",
    mass=4.5,
    inertia=(0.1, 0.1, 0.001),
    position=1.10,
    radius=0.05,
)

print(f"payload.mass = {payload.mass} kg")
print(f"payload.position = {payload.position} m")
print(f"payload.surfaces (before add_surface) = {payload.surfaces}")

In [ ]:
# add_surface requires radius to be set
no_radius_payload = Deployable(
    name="no_radius_payload", mass=1.0, inertia=(0, 0, 0), position=0.5
)
nose = NoseCone(length=0.2, kind="vonKarman", base_radius=0.05, rocket_radius=0.05)
try:
    no_radius_payload.add_surface(nose, position=0.1)
except ValueError as error:
    print(f"Raised as expected: {error}")

In [ ]:
# add_surface and free_rocket are mutually exclusive
payload.add_surface(nose, position=0.1)
print(f"payload.surfaces (after add_surface) = {payload.surfaces}")

## `MultiStageRocket`: composing a stage + deployable into one flight-ready `Rocket`

`flight_rocket()` composes the wrapped stage's mass/inertia/CoM with every deployable still aboard (parallel axis theorem), attaches the stage's motor, and reuses its aerodynamic surfaces and drag curve. Only a single active stage is supported so far — multi-stage composition (booster + sustainer together) is a later commit.

In [ ]:
vehicle = MultiStageRocket(stages=[booster])
deployable = vehicle.add_deployable(
    name="payload", mass=4.5, inertia=(0.1, 0.1, 0.001), position=1.10
)

flight_rocket = vehicle.flight_rocket(
    active_stages=(booster,), carried_deployables=(deployable,)
)

# Hand-computed weighted average, independent of flight_rocket's own
# code path — same check used in tests/unit/rocket/test_multistage.py
expected_mass = calisto.mass + 4.5
expected_center_of_mass = (
    calisto.mass * calisto.center_of_mass_without_motor + 4.5 * 1.10
) / expected_mass

print(f"flight_rocket.mass = {flight_rocket.mass:.4f} kg (expected {expected_mass:.4f})")
print(
    f"flight_rocket.center_of_mass_without_motor = {flight_rocket.center_of_mass_without_motor:.4f} m "
    f"(expected {expected_center_of_mass:.4f})"
)

assert abs(flight_rocket.mass - expected_mass) < 1e-9
assert abs(
    flight_rocket.center_of_mass_without_motor - expected_center_of_mass
) < 1e-9

## Multiple active stages: a booster carrying an inert sustainer

When more than one stage is still attached, `flight_rocket()` treats `active_stages[0]` as the currently firing stage (its motor becomes the composed `Rocket`'s own motor) and every other active stage as inert cargo riding along - contributing its *full* current mass (structure + motor + unburned propellant), since its own motor clock hasn't started yet.

Each stage's own `Rocket` coordinate system is assumed to already be expressed in a shared stack frame - positions are used as-is. Deriving stack positions automatically from `interstage_lengths` and each stage's physical extent isn't implemented yet.

In [ ]:
from rocketpy.motors.point_mass_motor import PointMassMotor

booster_rocket = Rocket(
    radius=0.1,
    mass=10.0,
    inertia=(1.0, 1.0, 0.01),
    power_off_drag=0.5,
    power_on_drag=0.6,
    center_of_mass_without_motor=0.0,
)
booster_rocket.add_motor(
    PointMassMotor(
        thrust_source=100, dry_mass=1.0, propellant_initial_mass=2.0, burn_time=1.0
    ),
    position=0.0,
)
two_stage_booster = Stage(name="booster", rocket=booster_rocket)

sustainer_rocket = Rocket(
    radius=0.08,
    mass=5.0,
    inertia=(0.5, 0.5, 0.005),
    power_off_drag=0.3,
    power_on_drag=0.4,
    center_of_mass_without_motor=2.0,
)
sustainer_rocket.add_motor(
    PointMassMotor(
        thrust_source=50, dry_mass=0.5, propellant_initial_mass=1.0, burn_time=1.0
    ),
    position=2.0,
)
sustainer = Stage(name="sustainer", rocket=sustainer_rocket)

two_stage_vehicle = MultiStageRocket(stages=[two_stage_booster, sustainer])

In [ ]:
stacked = two_stage_vehicle.flight_rocket(
    active_stages=(two_stage_booster, sustainer)
)
print(f"stacked.mass = {stacked.mass} kg  (booster structure + full inert sustainer)")
print(f"stacked.center_of_mass_without_motor = {stacked.center_of_mass_without_motor:.4f} m")
print(f"stacked.motor is booster_rocket.motor -> {stacked.motor is booster_rocket.motor}")

after_separation = two_stage_vehicle.flight_rocket(active_stages=(sustainer,))
print(f"\nafter_separation.mass = {after_separation.mass} kg  (sustainer structure only)")
print(f"after_separation.motor is sustainer_rocket.motor -> "
      f"{after_separation.motor is sustainer_rocket.motor}")

assert stacked.mass == 10.0 + (5.0 + 0.5 + 1.0)
assert after_separation.mass == 5.0

## `Mission`: the degenerate single-flight case

`Mission` is the orchestrator that will eventually walk a vehicle's separation/ejection events and run one `Flight` per configuration with state handoff between them. Right now only the simplest case is implemented: a single-stage vehicle with no deployables degenerates to a thin wrapper around one `Flight` - confirming the invariant stated in `mission_multistage_design.md`. Anything else raises `NotImplementedError` for now.

A plain `Rocket` is sugar for a single-stage `MultiStageRocket` with nothing separable, so `Mission` accepts either directly.

In [ ]:
from rocketpy import Environment
from rocketpy.simulation.mission import Mission

mission = Mission(
    vehicle=calisto,
    environment=Environment(),
    rail_length=5.2,
    inclination=85,
    heading=0,
)

print(f"bodies flown: {list(mission.flights.keys())}")
print(f"flights for 'stage_1': {mission.flights['stage_1']}")
print("\ntimeline:")
for time, event in mission.timeline:
    print(f"  t={time:8.3f}s  {event}")

## `Mission`: a real two-stage flight

With two stages, `Mission` runs three `Flight`s: the full stack (booster firing), then the spent booster falling away on its own, and the sustainer continuing on its own. Separation and ignition timing are deterministic - computed ahead of time from `booster.burn_out_time + booster.separation` and `sustainer.ignition_delay` - there's no generic mid-flight event solver in RocketPy today, only `Flight`'s `max_time` and `terminate_on_apogee`.

The handoff between stack and children follows `mission_multistage_design.md`'s `_handoff_state`: the parent's ending position/velocity is transformed by the body-frame offset between the parent and child center-of-dry-mass, rotated into the inertial frame, plus (for the child) its momentum-conserving share of `separation_delta_v`. The sustainer's own motor is additionally re-anchored in time so it ignites at its actual mission time, not its own local t=0 (`_shift_motor_ignition`).

In [ ]:
from rocketpy.motors.point_mass_motor import PointMassMotor

ts_booster_rocket = Rocket(
    radius=0.1,
    mass=10.0,
    inertia=(1.0, 1.0, 0.01),
    power_off_drag=0.5,
    power_on_drag=0.6,
    center_of_mass_without_motor=0.0,
)
ts_booster_rocket.add_motor(
    PointMassMotor(
        thrust_source=400, dry_mass=1.0, propellant_initial_mass=2.0, burn_time=1.0
    ),
    position=0.0,
)
# separation: burns out at t=1.0s, jettisoned 0.5s later
ts_booster = Stage(name="booster", rocket=ts_booster_rocket, separation=0.5)

ts_sustainer_rocket = Rocket(
    radius=0.08,
    mass=5.0,
    inertia=(0.5, 0.5, 0.005),
    power_off_drag=0.3,
    power_on_drag=0.4,
    center_of_mass_without_motor=2.0,
)
ts_sustainer_rocket.add_motor(
    PointMassMotor(
        thrust_source=200, dry_mass=0.5, propellant_initial_mass=1.0, burn_time=1.0
    ),
    position=2.0,
)
# ignites immediately on separation
ts_sustainer = Stage(name="sustainer", rocket=ts_sustainer_rocket, ignition_delay=0.0)

two_stage_vehicle = MultiStageRocket(stages=[ts_booster, ts_sustainer])

In [ ]:
two_stage_mission = Mission(
    vehicle=two_stage_vehicle,
    environment=Environment(),
    rail_length=1.0,
    inclination=90,
    heading=0,
    max_time=30,
)

print(f"bodies flown: {list(two_stage_mission.flights.keys())}")
print(
    "'booster' flights (stack, then booster alone): "
    f"{len(two_stage_mission.flights['booster'])}"
)
print(
    "'sustainer' flights (stack, then sustainer alone): "
    f"{len(two_stage_mission.flights['sustainer'])}"
)
print("\ntimeline:")
for time, event in two_stage_mission.timeline:
    print(f"  t={time:8.3f}s  {event}")

## Not built yet

Deliberately out of scope for the commits so far (see the roadmap in `mission_multistage_design.md`):

- Deriving stack positions from `interstage_lengths` and each stage's physical extent (stages are currently assumed to already share one coordinate frame)
- `Mission` for more than two stages
- `Mission` deployable ejection (apogee-triggered, via `terminate_on_apogee`)
- Deterministic time-based separation (motor burnout + delay) and apogee-triggered deployable ejection
- `StochasticMission`

Each lands as its own commit with its own tests; this notebook grows alongside them.